In [1]:
import numpy as np
import pandas as pd
import fastf1 as f1

In [2]:
circuits = pd.read_csv('data/circuits.csv')
constructor_results = pd.read_csv('data/constructor_results.csv')
constructor_standings = pd.read_csv('data/constructor_standings.csv')
constructors = pd.read_csv('data/constructors.csv')
driver_standings = pd.read_csv('data/driver_standings.csv')
drivers = pd.read_csv('data/drivers.csv')
lap_times = pd.read_csv('data/lap_times.csv')
pit_stops = pd.read_csv('data/pit_stops.csv')
qualifying = pd.read_csv('data/qualifying.csv')
races = pd.read_csv('data/races.csv')
results = pd.read_csv('data/results.csv')
sprint_results = pd.read_csv('data/sprint_results.csv')
status = pd.read_csv('data/status.csv')
hungary = f1.get_session(2024, 13, "Race")
belgium = f1.get_session(2024, 14, "Race")

req         WARNING 	DEFAULT CACHE ENABLED! (291.12 MB) /Users/jestonlewis/Library/Caches/fastf1


In [3]:
hungary.load()
belgium.load()

core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.4.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '44', '16', '1', '55', '11', '63', '22', '18', '14', '3', '27', '23', '20', '77', '2', '31', '24', '10']
core           INFO 	Loading data for Belgian Grand Prix - R

In [4]:
results = results.drop(results[results.raceId < 989].index)
races = races.drop(races[races.year < 2018].index)

In [5]:
final = pd.merge(results, races, on="raceId")
final = pd.merge(final, circuits, on="circuitId")
final = pd.merge(final, drivers, on="driverId")

In [6]:
final.drop(columns=["resultId", "number_x", "positionText", "positionOrder", "points", "laps", "time_x", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "round", "name_x", "url_x", "fp1_date", "fp1_time", "fp2_date", "fp2_time", "fp3_date", "fp3_time", "quali_date", "quali_time", "sprint_date", "sprint_time", "name_y", "location", "country", "lat", "lng", "url_y", "number_y", "code", "forename", "surname", "dob", "nationality", "url", "alt"], inplace=True)

final = pd.merge(final, constructors, on="constructorId")

In [7]:
final.drop(columns=["name", "nationality", "url"], inplace=True)

In [8]:
final.rename(columns={"time_y":"time"}, inplace=True)

In [9]:
races = [belgium, hungary]

In [10]:
for race in races:
    race_results = race.results
    race_results.drop(columns=["DriverNumber", "BroadcastName", "Abbreviation", "TeamColor", "FirstName", "LastName", "FullName", "HeadshotUrl", "CountryCode", "ClassifiedPosition", "Q1", "Q2", "Q3", "Time", "Status"], inplace=True)
    race_results = pd.merge(race_results, drivers, left_on="DriverId", right_on="driverRef")
    race_results["year"] = race.session_info["StartDate"].date().year
    race_results["date"] = race.session_info["StartDate"].date().strftime("%Y-%m-%d")
    race_results["time"] = race.session_info["StartDate"].time().strftime("%H:%M")
    if race.session_info["Meeting"]["Circuit"]["ShortName"] == "Spa-Francorchamps":
        # TODO come up with better way to add circuit names
        race_results["circuitRef"] = "spa"
        race_results["raceId"] = 1134
    else:
        race_results["circuitRef"] = "hungaroring"
        race_results["raceId"] = 1133
    race_results = pd.merge(race_results, circuits, on="circuitRef")
    race_results = pd.merge(race_results, constructors, left_on="TeamId", right_on="constructorRef")
    race_results.drop(columns=["DriverId", "TeamName", "TeamId", "Points", "number", "code", "country", "lat", "lng", "alt", "url_y", "name_y", "nationality_y", "url", "forename", "surname", "dob", "nationality_x", "url_x", "name_x", "location"], inplace=True)
    race_results.rename(columns={"Position":"position", "GridPosition":"grid"}, inplace=True)
    final = pd.concat([final, race_results], axis=0)

In [11]:
final.replace(to_replace="\\N", value=np.nan, inplace=True)

In [12]:
final.drop(columns=["driverId", "constructorId", "circuitId"], inplace=True)

In [13]:
final["circuit_code"] = final["circuitRef"].astype("category").cat.codes
final["driver_code"] = final["driverRef"].astype("category").cat.codes
final["constructor_code"] = final["constructorRef"].astype("category").cat.codes

In [14]:
final["position"] = final["position"].astype(float)

In [15]:
final["pos_delta"] = final["grid"] - final["position"]

In [16]:
final

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta
0,989,3.0,1.0,2018,2018-03-25,05:10:00,albert_park,vettel,ferrari,0,36,4,2.0
1,989,1.0,2.0,2018,2018-03-25,05:10:00,albert_park,hamilton,mercedes,0,11,8,-1.0
2,989,2.0,3.0,2018,2018-03-25,05:10:00,albert_park,raikkonen,ferrari,0,27,4,-1.0
3,989,8.0,4.0,2018,2018-03-25,05:10:00,albert_park,ricciardo,red_bull,0,28,11,4.0
4,989,10.0,5.0,2018,2018-03-25,05:10:00,albert_park,alonso,mclaren,0,2,7,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,1133,12.0,16.0,2024,2024-07-21,15:00,hungaroring,bottas,sauber,6,4,13,-4.0
16,1133,14.0,17.0,2024,2024-07-21,15:00,hungaroring,sargeant,williams,6,31,15,-3.0
17,1133,19.0,18.0,2024,2024-07-21,15:00,hungaroring,ocon,alpine,6,23,2,1.0
18,1133,18.0,19.0,2024,2024-07-21,15:00,hungaroring,zhou,sauber,6,37,13,-1.0


In [17]:
final.to_csv("data/final.csv", index=False)

In [18]:
def rolling_finish_avg(group, cols, new_cols):
    group = group.sort_values("raceId")
    rolling_stats = group[cols].rolling(3, closed="left").mean()
    group[new_cols] = rolling_stats
    return group

In [19]:
cols = ["grid", "position", "pos_delta"]
new_cols = [f"{c}_rolling" for c in cols]

In [20]:
final_rolling = final.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")

/var/folders/hk/ffpx1y0s2cn133v15g3yks080000gn/T/ipykernel_12890/536465426.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rolling = final.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")


In [21]:
final_rolling = final_rolling.sort_values(["raceId", "position"])

In [22]:
final_rolling.index = range(final_rolling.shape[0])

In [23]:
final_rolling.to_csv("data/final_rolling.csv", index=False)

In [24]:
final_rolling

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta,grid_rolling,position_rolling,pos_delta_rolling
0,989,3.0,1.0,2018,2018-03-25,05:10:00,albert_park,vettel,ferrari,0,36,4,2.0,NaN,NaN,NaN
1,989,1.0,2.0,2018,2018-03-25,05:10:00,albert_park,hamilton,mercedes,0,11,8,-1.0,NaN,NaN,NaN
2,989,2.0,3.0,2018,2018-03-25,05:10:00,albert_park,raikkonen,ferrari,0,27,4,-1.0,NaN,NaN,NaN
3,989,8.0,4.0,2018,2018-03-25,05:10:00,albert_park,ricciardo,red_bull,0,28,11,4.0,NaN,NaN,NaN
4,989,10.0,5.0,2018,2018-03-25,05:10:00,albert_park,alonso,mclaren,0,2,7,5.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2774,1134,20.0,16.0,2024,2024-07-28,15:00,spa,tsunoda,rb,25,34,10,4.0,12.333333,11.000000,1.333333
2775,1134,18.0,17.0,2024,2024-07-28,15:00,spa,sargeant,williams,25,31,15,1.0,15.000000,15.666667,-0.666667
2776,1134,16.0,18.0,2024,2024-07-28,15:00,spa,hulkenberg,haas,25,12,6,-2.0,8.666667,8.333333,0.333333
2777,1134,19.0,19.0,2024,2024-07-28,15:00,spa,zhou,sauber,25,37,13,0.0,17.333333,18.000000,-0.666667
